# 🤖 Agente IA Avanzado — Helpdesk DTI UDEP

Bot de Telegram con inteligencia artificial avanzada para la Dirección de Tecnologías de la Información.

## 🛠️ Funcionalidades
| Módulo | Descripción | Herramienta |
|---|---|---|
| 🎙️ Audio → Texto | Transcripción de mensajes de voz | Gemini 1.5 Flash (gratis) |
| 🧠 IA Conversacional | Respuestas inteligentes en lenguaje natural | OpenRouter (free/pago) |
| 🔮 Pronóstico de carga | Predice volumen de tickets por hora/día | Lógica estadística local |
| 🏷️ Clasificación automática | Categoriza tickets con IA | Gemini 1.5 Flash |
| 📊 Análisis de comportamiento | Patrones de uso, usuarios frecuentes | Pandas + Matplotlib |
| 🌦️ Clima | Condiciones meteorológicas por ciudad | wttr.in (API libre) |
| 💱 Tipo de cambio | Conversión USD/PEN y otras monedas | exchangerate-api (gratis) |
| 🔍 Búsqueda web | Información en tiempo real | DuckDuckGo (gratis) |
| 📄 Reportes PDF | Generación de informes administrativos | ReportLab (local) |
| 📈 Gráficos avanzados | Visualizaciones estadísticas | Matplotlib (local) |
| 🚨 Alertas de prioridad | Notificación automática de tickets críticos | Telegram Bot API |
| 🏆 Ranking de categorías | Top incidencias más frecuentes | Cálculo local |

---
## 🔑 Accesos que debes crear

1. **TOKEN de Telegram Bot** → Habla con [@BotFather](https://t.me/BotFather) en Telegram → `/newbot`
2. **Google AI Studio API Key (Gemini)** → Ve a [aistudio.google.com](https://aistudio.google.com) → *Get API Key* → crea una clave gratuita
3. **OpenRouter API Key** → Provista por el instructor (`sk-or-v1-...`)
4. **Guardar en Colab Secrets** → Menú 🔑 → agrega:
   - `TOKEN_DTI` → token de tu bot de Telegram
   - `GEMINI_API_KEY` → tu clave de Google AI Studio
   - `OPENROUTER_API_KEY` → clave de OpenRouter

---

## Celda 1 — Instalación de Dependencias

In [6]:
# ── Limpiar instalaciones anteriores ─────────────────────────────
!pip uninstall -y langchain langchain-openai langchain-community langchain-core

# ── Dependencias generales ───────────────────────────────────────
!pip install -q pyTelegramBotAPI matplotlib pillow reportlab
!pip install -q google-generativeai
!pip install -q duckduckgo-search ddgs
!pip install -q requests pandas numpy scipy
!pip install -q pydub

# ── Instalar versiones COMPATIBLES de LangChain ─────────────────
!pip install -q \
langchain==0.1.16 \
langchain-core==0.1.42 \
langchain-community==0.0.32 \
langchain-openai==0.1.3

# ── FFmpeg ───────────────────────────────────────────────────────
!apt-get install -y ffmpeg

print("✅ Todas las dependencias instaladas correctamente.")

Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-openai 1.2.1
Uninstalling langchain-openai-1.2.1:
  Successfully uninstalled langchain-openai-1.2.1
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 16.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-checkpoint 4.1.0 requires langchain-core>=0.2.38, but you have langchain-core 0.1.42 which is incompatible.
langgraph 1.2.0 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.1.42 wh

## Celda 2 — Importaciones y Configuración

In [9]:
import os, math, random, logging, tempfile
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use('Agg')

from io import BytesIO
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from scipy import stats

import telebot
from telebot.types import InlineKeyboardMarkup, InlineKeyboardButton

from PIL import Image, ImageDraw, ImageFont

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Table,
    TableStyle,
    Paragraph,
    Spacer
)

from reportlab.lib import colors
from reportlab.lib.styles import (
    getSampleStyleSheet,
    ParagraphStyle
)

import google.generativeai as genai

# ── LangChain ─────────────────────────────────────────────
from langchain.agents import AgentExecutor, create_react_agent

from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate

from langchain_openai import ChatOpenAI

from langchain_community.utilities import (
    DuckDuckGoSearchAPIWrapper
)

from google.colab import userdata

print("✅ Importaciones completadas.")

✅ Importaciones completadas.


## Celda 3 — Cargar Claves desde Colab Secrets

In [3]:
# ── Cargar claves desde Colab Secrets ────────────────────────────────────────
# IMPORTANTE: ve a 🔑 (Secrets) en el menú lateral y agrega:
#   TOKEN_DTI          → Token de tu bot de Telegram (@BotFather)
#   GEMINI_API_KEY     → Clave de Google AI Studio (aistudio.google.com)
#   OPENROUTER_API_KEY → Clave provista por el instructor

TELEGRAM_TOKEN     = userdata.get("TOKEN_DTI")
GEMINI_API_KEY     = userdata.get("GEMINI_API_KEY")
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Configurar Gemini
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-1.5-flash")

# Inicializar bot de Telegram
bot = telebot.TeleBot(TELEGRAM_TOKEN)
logging.basicConfig(level=logging.INFO)

print("✅ Claves cargadas y servicios inicializados.")
print(f"   Bot Token: ...{TELEGRAM_TOKEN[-8:]}")
print(f"   Gemini Key: ...{GEMINI_API_KEY[-6:]}")

✅ Claves cargadas y servicios inicializados.
   Bot Token: ...ghZ3aCPs
   Gemini Key: ...DnYPKc


## Celda 4 — Arquitectura de Datos

In [5]:
# ── Base de datos en memoria ──────────────────────────────────────────────────
tickets:        dict = {}
next_ticket_id: int  = 1000
user_roles:     dict = {}    # {chat_id: str}
user_sessions:  dict = {}    # {chat_id: dict}
chat_histories: dict = {}    # {chat_id: list} para memoria conversacional del agente IA

ADMIN_CODE = "767574"

# ── Catálogos del sistema ─────────────────────────────────────────────────────
CATEGORIAS = {
    "Accesos y Cuentas": [
        "Olvido de contraseña", "Cuenta bloqueada", "Problemas con login",
        "Activacion de cuenta nueva", "Acceso a plataformas UDEP",
        "Problemas con autenticacion 2FA",
    ],
    "Red e Internet": [
        "Falla de red WiFi", "Senal debil o intermitente",
        "No conexion a internet", "Problemas con red cableada (LAN)",
    ],
    "Hardware": [
        "Computadora no enciende", "Pantalla danada", "Sobrecalentamiento",
        "Ruido extrano en equipo", "Problemas con impresoras",
        "Proyectores o equipos de aula danados",
    ],
    "Software": [
        "Software no abre", "Programa se cierra inesperadamente",
        "Error de instalacion", "Licencias vencidas",
        "Actualizaciones fallidas", "Virus o malware", "Lentitud del sistema",
    ],
    "Impresoras y Perifericos": [
        "Impresora no imprime", "Atasco de papel",
        "Baja calidad de impresion", "Problemas con scanner",
        "No detecta el dispositivo",
    ],
    "Plataformas Academicas": [
        "Problemas con aula virtual", "No acceso a cursos",
        "Error al subir tareas", "Fallas en evaluaciones online",
        "Problemas con videoclases (Zoom/Teams)",
    ],
    "Infraestructura de Aula": [
        "Falta de internet en aula", "Proyector no funciona",
        "Equipos del aula danados", "Problemas electricos (enchufes)",
        "Falta de cables/adaptadores",
    ],
    "Correo Institucional": [
        "No acceso al correo", "No envio/recepcion de correos",
        "Correos en spam", "Configuracion en celular",
        "Problemas con adjuntos",
    ],
    "Seguridad Informatica": [
        "Sospecha de hackeo", "Phishing", "Acceso no autorizado",
        "Archivos sospechosos", "Robo de informacion",
    ],
    "Otros / General": [
        "Consulta general", "Soporte tecnico no clasificado",
        "Requerimientos especiales",
    ],
}

AMBIENTES   = ["Edificio 80", "Principal", "Edificio A", "IME", "CPA",
               "FCOM", "Ingenieria Civil", "Educacion", "Gobierno",
               "Quimica", "Derecho", "Edificio E", "Arquitectura", "Carpa",
               "Cafeta", "Otro"]
PRIORIDADES = ["Inmediata", "Urgente", "Normal", "No urgente"]
ROLES       = ["Estudiante", "Profesor", "Personal Administrativo", "Administrador TI"]

print("✅ Arquitectura de datos lista.")

✅ Arquitectura de datos lista.


## Celda 5 — Datos de Prueba

In [6]:
# ── Poblar con tickets de prueba ──────────────────────────────────────────────
def _seed_demo():
    global next_ticket_id
    cats = list(CATEGORIAS.keys())
    descripciones = [
        "No puedo ingresar al sistema desde esta manana.",
        "La computadora hace un ruido extrano al encenderse.",
        "La red WiFi del laboratorio cae constantemente.",
        "No puedo subir tareas al aula virtual.",
        "El proyector del salon 201 no enciende.",
        "Necesito instalar AutoCAD pero da error de licencia.",
        "Recibi un correo sospechoso con enlace extrano.",
        "La impresora del departamento no imprime en color.",
        "Mi correo institucional no recibe mensajes externos.",
        "La pantalla de mi laptop tiene rayas horizontales.",
        "El sistema se pone lento a las 10am cuando hay clase.",
        "No puedo conectarme a Teams para la clase virtual.",
        "Necesito activar mi cuenta de estudiante nuevo.",
        "Falta un cable HDMI en el aula de Derecho.",
        "El antivirus bloquea un programa que necesito.",
    ]
    for i in range(20):
        tid = next_ticket_id
        next_ticket_id += 1
        cat   = random.choice(cats)
        fecha = (
            datetime.now()
            - timedelta(days=random.randint(0, 13))
            - timedelta(hours=random.randint(6, 20), minutes=random.randint(0, 59))
        )
        tickets[tid] = {
            "id":          tid,
            "chat_id":     random.choice([111111, 222222, 333333, 444444]),
            "estado":      random.choice(["Pendiente", "Pendiente", "Atendido", "En proceso"]),
            "fecha":       fecha,
            "prioridad":   random.choice(["Normal", "Normal", "Urgente", "No urgente", "Inmediata"]),
            "categoria":   cat,
            "subcategoria":random.choice(CATEGORIAS[cat]),
            "descripcion": random.choice(descripciones),
            "ambiente":    random.choice(AMBIENTES),
            "aula":        f"Aula {random.randint(100, 310)}",
            "rol_creador": random.choice(["Estudiante", "Profesor", "Personal Administrativo"]),
            "tiempo_resolucion": random.choice([None, None, random.randint(30, 480)]),  # minutos
        }

_seed_demo()
print(f"✅ {len(tickets)} tickets de prueba generados.")

✅ 20 tickets de prueba generados.


## Celda 6 — Módulo de IA: Clasificación Automática con Gemini

In [10]:
# ── Clasificación automática de tickets con Gemini 1.5 Flash ─────────────────
def clasificar_ticket_con_ia(descripcion: str) -> dict:
    """
    Usa Gemini 1.5 Flash para clasificar automáticamente un ticket
    a partir de su descripción en lenguaje natural.
    Devuelve: {categoria, subcategoria, prioridad, resumen}
    """
    cats_str = "\n".join([f"- {k}: {', '.join(v[:3])}..." for k, v in CATEGORIAS.items()])
    prompt = f"""
Eres un asistente de soporte IT de la Universidad de Piura (UDEP).
Analiza esta descripción de incidente y clasifícala.

Descripción del usuario: "{descripcion}"

Categorías disponibles:
{cats_str}

Responde SOLO con este formato JSON exacto (sin markdown, sin explicaciones):
{{"categoria": "nombre exacto de la categoria", "subcategoria": "subcategoria especifica", "prioridad": "Normal|Urgente|Inmediata|No urgente", "resumen": "resumen en 10 palabras"}}
"""
    try:
        response = gemini_model.generate_content(prompt)
        import json
        texto = response.text.strip().replace("```json", "").replace("```", "")
        return json.loads(texto)
    except Exception as e:
        return {
            "categoria": "Otros / General",
            "subcategoria": "Soporte tecnico no clasificado",
            "prioridad": "Normal",
            "resumen": descripcion[:50]
        }

# ── Transcripción de audio con Gemini 1.5 Flash ───────────────────────────────
def transcribir_audio_gemini(file_bytes: bytes, mime_type: str = "audio/ogg") -> str:
    """
    Transcribe un mensaje de voz usando Gemini 1.5 Flash.
    Acepta audio OGG (formato nativo de Telegram).
    """
    try:
        # Subir el archivo de audio a la API de Gemini
        with tempfile.NamedTemporaryFile(suffix=".ogg", delete=False) as tmp:
            tmp.write(file_bytes)
            tmp_path = tmp.name

        audio_file = genai.upload_file(tmp_path, mime_type=mime_type)

        response = gemini_model.generate_content([
            audio_file,
            "Transcribe exactamente lo que dice este audio en español. "
            "Si no hay voz clara, responde: [Audio sin contenido claro]. "
            "Solo devuelve la transcripción, sin explicaciones adicionales."
        ])

        os.unlink(tmp_path)
        return response.text.strip()
    except Exception as e:
        return f"[Error al transcribir audio: {str(e)[:60]}]"

# ── Respuesta conversacional con Gemini (para el agente IA libre) ─────────────
def respuesta_gemini_conversacional(cid: int, mensaje: str) -> str:
    """
    Mantiene una conversación multi-turno con Gemini 1.5 Flash.
    Guarda historial por usuario para dar contexto.
    """
    if cid not in chat_histories:
        chat_histories[cid] = []

    # Limitar historial a últimas 10 interacciones para no superar contexto
    historial = chat_histories[cid][-10:]

    system_prompt = """
Eres el Asistente Virtual de TI de la Universidad de Piura (UDEP).
Ayudas a estudiantes, profesores y personal administrativo con:
- Problemas técnicos de computadoras, redes, software, impresoras
- Acceso a plataformas académicas (aula virtual, correo, etc.)
- Guías paso a paso para resolver problemas comunes
- Información sobre el estado de incidentes
Responde siempre en español, de forma clara, amigable y concisa.
Si no puedes resolver el problema, indica que se cree un ticket formal.
"""

    # Construir conversación
    conversation = [system_prompt]
    for h in historial:
        conversation.append(f"Usuario: {h['user']}")
        conversation.append(f"Asistente: {h['assistant']}")
    conversation.append(f"Usuario: {mensaje}")

    try:
        full_prompt = "\n".join(conversation) + "\n\nAsistente:"
        response = gemini_model.generate_content(full_prompt)
        respuesta = response.text.strip()

        # Guardar en historial
        chat_histories[cid].append({"user": mensaje, "assistant": respuesta})
        return respuesta
    except Exception as e:
        return f"Lo siento, hubo un error al procesar tu consulta. Por favor intenta de nuevo."

print("✅ Módulos de IA (Gemini) cargados.")

✅ Módulos de IA (Gemini) cargados.


## Celda 7 — Agente LangChain con OpenRouter (Herramientas Externas)

In [12]:
# ── Herramientas del Agente LangChain ────────────────────────────────────────
# 📊 Estadísticas de tickets en tiempo real
def stats_tool(query: str) -> str:
    """Consulta estadísticas del sistema de tickets en tiempo real."""
    total    = len(tickets)
    pend     = sum(1 for t in tickets.values() if t["estado"] == "Pendiente")
    atend    = sum(1 for t in tickets.values() if t["estado"] == "Atendido")
    en_proc  = sum(1 for t in tickets.values() if t["estado"] == "En proceso")
    urgentes = sum(1 for t in tickets.values() if t["prioridad"] in ["Urgente", "Inmediata"])

    cats = Counter(t["categoria"] for t in tickets.values())
    top_cat = cats.most_common(3)
    top_str = ", ".join([f"{c[0]} ({c[1]})" for c in top_cat])

    return (f"📊 Tickets totales: {total} | Pendientes: {pend} | "
            f"En proceso: {en_proc} | Atendidos: {atend} | "
            f"Urgentes/Inmediatos: {urgentes} | "
            f"Top categorías: {top_str}")

# 🔮 Pronóstico de carga
def forecast_tool(query: str) -> str:
    """Pronostica el volumen esperado de tickets para las próximas horas/días."""
    # Calcular distribución por hora del día
    hora_conteo = Counter(t["fecha"].hour for t in tickets.values())
    hora_actual = datetime.now().hour

    # Promedio de tickets por hora
    if hora_conteo:
        total_dias = max((datetime.now() - min(t["fecha"] for t in tickets.values())).days + 1, 1)
        prom_diario = len(tickets) / total_dias

        # Horas pico históricas
        horas_pico = [h for h, c in hora_conteo.most_common(3)]
        horas_str  = ", ".join([f"{h:02d}:00h" for h in sorted(horas_pico)])

        # Tendencia: ¿está subiendo o bajando?
        ultimos_7 = [t for t in tickets.values() if t["fecha"] > datetime.now() - timedelta(days=7)]
        primeros_7 = [t for t in tickets.values() if t["fecha"] <= datetime.now() - timedelta(days=7)]
        tendencia = "📈 creciente" if len(ultimos_7) > len(primeros_7) else "📉 decreciente"

        return (f"🔮 Pronóstico de carga:\n"
                f"  • Promedio diario histórico: {prom_diario:.1f} tickets/día\n"
                f"  • Horas pico: {horas_str}\n"
                f"  • Tendencia semanal: {tendencia}\n"
                f"  • Tickets últimos 7 días: {len(ultimos_7)}\n"
                f"  • Recomendación: {'Mayor disponibilidad en horas pico' if hora_actual in horas_pico else 'Carga normal esperada'}")
    return "Insuficiente datos históricos para pronóstico."

# ── Configurar herramientas LangChain ────────────────────────────────────────
langchain_tools = [
    Tool(name="Estadisticas",func=stats_tool,     description="Estadísticas actuales del sistema de tickets de soporte IT"),
    Tool(name="Pronostico",  func=forecast_tool,  description="Pronóstico de carga de tickets: cuándo se esperan más incidencias"),
]

# ── Crear el agente LangChain con OpenRouter ──────────────────────────────────
AGENT_PROMPT_TEMPLATE = """
Eres el Agente de IA del Helpdesk DTI de la Universidad de Piura (UDEP).
Tienes acceso a herramientas para dar respuestas precisas y útiles.

Herramientas disponibles:
{tools}

Formato OBLIGATORIO:
Pregunta: {input}
Thought: razona qué herramienta usar o si puedes responder directamente
Action: nombre de herramienta de [{tool_names}]
Action Input: entrada para la herramienta
Observation: resultado de la herramienta
... (repite si necesitas más herramientas)
Thought: ya tengo suficiente información
Final Answer: respuesta completa, clara y útil en español

Reglas:
- Usa herramientas cuando necesites datos en tiempo real
- No inventes datos, siempre verifica
- Sé conciso pero completo
- Responde siempre en español

{agent_scratchpad}
"""

agent_prompt = PromptTemplate.from_template(AGENT_PROMPT_TEMPLATE)

llm_openrouter = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openrouter/auto",   # usa el mejor modelo disponible
    temperature=0,
    max_tokens=1200
)

react_agent    = create_react_agent(llm_openrouter, langchain_tools, agent_prompt)
agent_executor = AgentExecutor(
    agent=react_agent,
    tools=langchain_tools,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=5,
    return_intermediate_steps=False
)

def consulta_agente_openrouter(pregunta: str) -> str:
    try:
        result = agent_executor.invoke({"input": pregunta})
        return str(result.get("output", "Sin respuesta."))
    except Exception as e:
        return f"Error en el agente: {str(e)[:100]}"

print("✅ Agente LangChain + OpenRouter configurado con", len(langchain_tools), "herramientas.")

✅ Agente LangChain + OpenRouter configurado con 2 herramientas.


## Celda 8 — Módulo de Análisis: Pronósticos, Comportamiento y Rankings

In [13]:
# ── Análisis avanzado de tickets ──────────────────────────────────────────────

def analisis_comportamiento() -> dict:
    """Analiza patrones de comportamiento en el sistema de tickets."""
    if not tickets:
        return {}

    df = pd.DataFrame(list(tickets.values()))
    df["hora"]     = df["fecha"].apply(lambda x: x.hour)
    df["dia_sem"]  = df["fecha"].apply(lambda x: x.weekday())
    df["dia_nombre"] = df["fecha"].apply(lambda x: ["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"][x.weekday()])

    # Horas pico
    horas_conteo = df["hora"].value_counts().sort_index()
    hora_pico    = int(df["hora"].value_counts().idxmax())

    # Día más activo
    dia_pico = df["dia_nombre"].value_counts().idxmax()

    # Tasa de resolución
    tasa_resol = (df["estado"] == "Atendido").mean() * 100

    # Tiempo promedio de resolución
    tiempos = df["tiempo_resolucion"].dropna()
    tpr = tiempos.mean() if len(tiempos) > 0 else 0

    # Tickets por rol
    por_rol = df["rol_creador"].value_counts().to_dict()

    # Categorías más frecuentes
    top_cats = df["categoria"].value_counts().head(5).to_dict()

    # Tickets urgentes/inmediatos pendientes
    criticos = df[(df["prioridad"].isin(["Urgente","Inmediata"])) &
                  (df["estado"] != "Atendido")]

    return {
        "total": len(df),
        "hora_pico": hora_pico,
        "dia_pico": dia_pico,
        "tasa_resolucion": round(tasa_resol, 1),
        "tiempo_prom_resolucion_min": round(tpr, 0),
        "por_rol": por_rol,
        "top_categorias": top_cats,
        "criticos_pendientes": len(criticos),
        "horas_conteo": horas_conteo.to_dict(),
    }

def pronostico_tickets_prox_dias(dias: int = 7) -> pd.DataFrame:
    """Genera pronóstico de tickets para los próximos N días usando tendencia lineal."""
    if len(tickets) < 5:
        return pd.DataFrame()

    # Agrupar tickets por día
    conteo_diario = defaultdict(int)
    for t in tickets.values():
        dia_key = t["fecha"].strftime("%Y-%m-%d")
        conteo_diario[dia_key] += 1

    fechas = sorted(conteo_diario.keys())
    y = np.array([conteo_diario[f] for f in fechas])
    x = np.arange(len(y))

    # Regresión lineal simple
    slope, intercept, r, p, se = stats.linregress(x, y)

    # Proyectar próximos N días
    ultimo_idx = len(y)
    pronostico_rows = []
    ultima_fecha = datetime.strptime(fechas[-1], "%Y-%m-%d")
    for i in range(1, dias + 1):
        fecha_p = ultima_fecha + timedelta(days=i)
        valor_p = max(0, round(slope * (ultimo_idx + i - 1) + intercept))
        pronostico_rows.append({
            "fecha": fecha_p.strftime("%d/%m"),
            "tickets_esperados": int(valor_p),
            "confianza": f"{min(abs(r)*100, 95):.0f}%"
        })

    return pd.DataFrame(pronostico_rows)

def ranking_categorias() -> list:
    """Genera ranking de categorías por frecuencia, urgencia y tiempo de resolución."""
    ranking = {}
    for t in tickets.values():
        cat = t["categoria"]
        if cat not in ranking:
            ranking[cat] = {"total": 0, "urgentes": 0, "resueltos": 0, "score": 0}
        ranking[cat]["total"] += 1
        if t["prioridad"] in ["Urgente", "Inmediata"]:
            ranking[cat]["urgentes"] += 1
        if t["estado"] == "Atendido":
            ranking[cat]["resueltos"] += 1

    # Calcular score ponderado (frecuencia + urgencia)
    for cat in ranking:
        r = ranking[cat]
        r["score"] = r["total"] * 1.0 + r["urgentes"] * 2.0
        r["pct_resolucion"] = round(r["resueltos"] / r["total"] * 100, 1) if r["total"] > 0 else 0

    return sorted(ranking.items(), key=lambda x: x[1]["score"], reverse=True)

print("✅ Módulos de análisis y pronóstico cargados.")

✅ Módulos de análisis y pronóstico cargados.


## Celda 9 — Gráficos Avanzados

In [14]:
# ── Generación de gráficos avanzados ─────────────────────────────────────────

def _grafico_pie(criterio: str) -> BytesIO:
    """Gráfico circular por estado, categoría o ambiente."""
    if criterio == "Estado":
        conteo = Counter(t["estado"] for t in tickets.values())
    elif criterio == "Categoria":
        conteo = Counter(t["categoria"] for t in tickets.values())
    else:
        conteo = Counter(t["ambiente"] for t in tickets.values())

    fig, ax = plt.subplots(figsize=(9, 7))
    wedges, texts, autotexts = ax.pie(
        conteo.values(), labels=conteo.keys(),
        autopct="%1.1f%%", startangle=140,
        pctdistance=0.82, wedgeprops=dict(width=0.6),
        colors=plt.cm.Set3.colors
    )
    for at in autotexts:
        at.set_fontsize(9)
    ax.set_title(f"Distribución de Tickets por {criterio}", fontsize=13, fontweight="bold", pad=18)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8)
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="PNG", bbox_inches="tight", dpi=130)
    plt.close()
    buf.seek(0)
    return buf

def _grafico_temporal() -> BytesIO:
    """Gráfico de líneas: evolución de tickets en la última semana."""
    creados   = Counter()
    resueltos = Counter()
    for t in tickets.values():
        dia = t["fecha"].strftime("%d/%m")
        creados[dia] += 1
        if t["estado"] == "Atendido":
            resueltos[dia] += 1

    dias   = sorted(set(list(creados) + list(resueltos)))
    c_vals = [creados.get(d, 0)   for d in dias]
    r_vals = [resueltos.get(d, 0) for d in dias]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.fill_between(dias, c_vals, alpha=0.12, color="#1e3c78")
    ax.fill_between(dias, r_vals, alpha=0.12, color="#e74c3c")
    ax.plot(dias, c_vals, marker="o", linewidth=2, label="Creados",   color="#1e3c78")
    ax.plot(dias, r_vals, marker="s", linewidth=2, label="Resueltos", color="#e74c3c")
    for x, y in zip(dias, c_vals):
        ax.annotate(str(y), (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
    ax.set_title("Evolución de Tickets — Historial", fontsize=13, fontweight="bold")
    ax.set_xlabel("Día")
    ax.set_ylabel("Cantidad")
    ax.legend(fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.xticks(rotation=25)
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="PNG", bbox_inches="tight", dpi=130)
    plt.close()
    buf.seek(0)
    return buf

def _grafico_horas_pico() -> BytesIO:
    """Gráfico de barras: distribución de tickets por hora del día."""
    hora_conteo = Counter(t["fecha"].hour for t in tickets.values())
    horas  = list(range(24))
    conteo = [hora_conteo.get(h, 0) for h in horas]

    colors_bar = ["#e74c3c" if c == max(conteo) else "#3498db" for c in conteo]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar([f"{h:02d}h" for h in horas], conteo, color=colors_bar, alpha=0.85, edgecolor="white")
    ax.set_title("📍 Tickets por Hora del Día (Horas Pico)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Hora")
    ax.set_ylabel("Cantidad de Tickets")
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    plt.xticks(rotation=45, fontsize=8)
    # Anotar barra más alta
    max_val = max(conteo)
    max_idx = conteo.index(max_val)
    ax.annotate(f"PICO\n{max_val}", xy=(max_idx, max_val),
                xytext=(max_idx, max_val + 0.3), ha="center",
                fontsize=9, color="#e74c3c", fontweight="bold")
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="PNG", bbox_inches="tight", dpi=130)
    plt.close()
    buf.seek(0)
    return buf

def _grafico_pronostico() -> BytesIO:
    """Gráfico de pronóstico para los próximos 7 días."""
    df_forecast = pronostico_tickets_prox_dias(7)
    if df_forecast.empty:
        return None

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(df_forecast["fecha"], df_forecast["tickets_esperados"],
           color="#2ecc71", alpha=0.8, edgecolor="white", label="Tickets esperados")
    ax.plot(df_forecast["fecha"], df_forecast["tickets_esperados"],
            marker="o", color="#27ae60", linewidth=2)
    for i, row in df_forecast.iterrows():
        ax.annotate(str(row["tickets_esperados"]),
                    (row["fecha"], row["tickets_esperados"]),
                    textcoords="offset points", xytext=(0, 6),
                    ha="center", fontsize=10, fontweight="bold")
    ax.set_title("🔮 Pronóstico de Tickets — Próximos 7 Días", fontsize=13, fontweight="bold")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Tickets Esperados")
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.legend(fontsize=10)
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="PNG", bbox_inches="tight", dpi=130)
    plt.close()
    buf.seek(0)
    return buf

def _grafico_ranking() -> BytesIO:
    """Gráfico de barras horizontales: ranking de categorías."""
    ranking = ranking_categorias()
    cats    = [r[0][:25] for r in ranking[:8]]
    scores  = [r[1]["score"] for r in ranking[:8]]
    urgentes = [r[1]["urgentes"] for r in ranking[:8]]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(cats[::-1], scores[::-1], color="#3498db", alpha=0.8, label="Score total")
    ax.barh(cats[::-1], urgentes[::-1], color="#e74c3c", alpha=0.8, label="Urgentes/Inmediatos")
    ax.set_title("🏆 Ranking de Categorías por Impacto", fontsize=13, fontweight="bold")
    ax.set_xlabel("Score (frecuencia + urgencia ponderada)")
    ax.legend(fontsize=9)
    ax.grid(True, axis="x", linestyle="--", alpha=0.4)
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="PNG", bbox_inches="tight", dpi=130)
    plt.close()
    buf.seek(0)
    return buf

print("✅ Módulo de gráficos avanzados cargado.")

✅ Módulo de gráficos avanzados cargado.


## Celda 10 — Generación de Comprobante PNG y Reporte PDF

In [15]:
# ── Comprobante PNG ───────────────────────────────────────────────────────────
def _png_ticket(ticket: dict) -> BytesIO:
    W, H = 640, 580
    img  = Image.new("RGB", (W, H), (245, 245, 250))
    draw = ImageDraw.Draw(img)

    # Encabezado
    draw.rectangle([0, 0, W, 74], fill=(30, 60, 120))
    draw.text((20, 12), "UDEP — Helpdesk DTI", fill="white")
    draw.text((20, 40), f"Comprobante de Ticket  #{ticket['id']}", fill=(190, 215, 255))

    # Badge de estado
    estado_colors = {"Atendido": (34, 139, 34), "Pendiente": (200, 100, 0), "En proceso": (41, 128, 185)}
    estado_color  = estado_colors.get(ticket.get("estado"), (100, 100, 100))

    campos = [
        ("Estado",        ticket.get("estado",       "-"), estado_color),
        ("Fecha",         ticket["fecha"].strftime("%d/%m/%Y  %H:%M"), (15, 15, 15)),
        ("Prioridad",     ticket.get("prioridad",    "Normal"), (15, 15, 15)),
        ("Categoria",     ticket.get("categoria",    "-"), (15, 15, 15)),
        ("Subcategoria",  ticket.get("subcategoria", "-"), (15, 15, 15)),
        ("Descripcion",   ticket.get("descripcion",  "-"), (15, 15, 15)),
        ("Ambiente",      ticket.get("ambiente",     "-"), (15, 15, 15)),
        ("Aula",          ticket.get("aula",         "-"), (15, 15, 15)),
        ("Creado por",    ticket.get("rol_creador",  "-"), (15, 15, 15)),
        ("Clasif. IA",    ticket.get("resumen_ia",   "Pendiente"), (50, 50, 150)),
    ]

    y = 94
    for label, valor, color in campos:
        draw.text((28, y),  f"{label}:",        fill=(100, 100, 100))
        draw.text((205, y), str(valor)[:58],     fill=color)
        draw.line([18, y + 23, W - 18, y + 23], fill=(215, 215, 220))
        y += 42

    # Pie
    draw.rectangle([0, H - 40, W, H], fill=(30, 60, 120))
    draw.text((18, H - 28), "Generado por AgenteIA DTI-UDEP  •  Helpdesk IT", fill=(175, 200, 255))

    buf = BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    return buf

# ── Reporte PDF Avanzado ──────────────────────────────────────────────────────
def _pdf_reporte(criterio: str) -> BytesIO:
    grupos = {}
    for t in tickets.values():
        key = t["estado"] if criterio == "Estado" else t.get(criterio.lower(), "Otros")
        grupos.setdefault(key, []).append(t)

    buf = BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, rightMargin=30, leftMargin=30, topMargin=30, bottomMargin=18)
    styles = getSampleStyleSheet()
    titulo_style = ParagraphStyle('T', parent=styles['Heading1'],
        alignment=1, textColor=colors.HexColor("#1E3C78"), spaceAfter=10)
    sub_style    = ParagraphStyle('S', parent=styles['Normal'],
        alignment=1, textColor=colors.grey, spaceAfter=20)

    analisis = analisis_comportamiento()

    elementos = [
        Paragraph("REPORTE AVANZADO DE TICKETS — DTI UDEP", titulo_style),
        Paragraph(
            f"Agrupado por: {criterio} | Generado: {datetime.now().strftime('%d/%m/%Y %H:%M')} | "
            f"Total: {analisis.get('total', 0)} | Tasa resolución: {analisis.get('tasa_resolucion', 0)}%",
            sub_style
        ),
        Spacer(1, 12),
    ]

    for grupo, lista in grupos.items():
        elementos.append(Paragraph(f"<b>{grupo.upper()}</b> ({len(lista)} tickets)", styles['Heading3']))
        data = [["ID", "Fecha", "Categoría", "Prioridad", "Estado", "Aula", "Clasif.IA"]]
        for t in lista:
            data.append([
                str(t["id"]),
                t["fecha"].strftime("%d/%m/%y"),
                t["categoria"][:18],
                t["prioridad"],
                t["estado"],
                t["aula"],
                t.get("resumen_ia", "-")[:15],
            ])
        tabla = Table(data, colWidths=[35, 55, 130, 60, 65, 60, 90])
        tabla.setStyle(TableStyle([
            ('BACKGROUND',    (0, 0), (-1,  0), colors.HexColor("#1E3C78")),
            ('TEXTCOLOR',     (0, 0), (-1,  0), colors.whitesmoke),
            ('ALIGN',         (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME',      (0, 0), (-1,  0), 'Helvetica-Bold'),
            ('FONTSIZE',      (0, 0), (-1, -1), 7),
            ('BOTTOMPADDING', (0, 0), (-1,  0), 8),
            ('BACKGROUND',    (0, 1), (-1, -1), colors.whitesmoke),
            ('GRID',          (0, 0), (-1, -1), 0.5, colors.grey),
            ('VALIGN',        (0, 0), (-1, -1), 'MIDDLE'),
        ]))
        elementos.extend([tabla, Spacer(1, 15)])

    doc.build(elementos)
    buf.seek(0)
    return buf

print("✅ Módulo de comprobantes y PDF cargado.")

✅ Módulo de comprobantes y PDF cargado.


## Celda 11 — Teclados, Mensajes y Utilidades de Sesión

In [16]:
# ── Textos de bienvenida ──────────────────────────────────────────────────────
BIENVENIDA_USUARIO = """
✅ Rol asignado: {rol}

Hola! Soy el Asistente de IA del Helpdesk DTI-UDEP.
Puedo ayudarte con incidentes técnicos y consultas generales.

📋 MENU DE OPCIONES:
━━━━━━━━━━━━━━━━━━━━
🆕 Reportar un incidente (con clasificación IA)
✏️ Modificar un ticket tuyo
🗑 Eliminar un ticket
🎫 Ver el estado de tu ticket
🤖 Consulta libre al Agente IA
🎙️ Envía un audio con tu consulta o incidente
━━━━━━━━━━━━━━━━━━━━
💡 Tip: Puedes enviarme un mensaje de voz y lo transcribiré automáticamente.
"""

BIENVENIDA_ADMIN = """
✅ Acceso concedido. Bienvenido, Administrador de TI!

Tienes acceso completo al sistema con IA avanzada.

📋 PANEL DE ADMINISTRADOR:
━━━━━━━━━━━━━━━━━━━━━━━━━
🔄 Actualizar estado de ticket
🔍 Consultar ticket / Reporte PDF
📊 Gráficos estadísticos avanzados
🔮 Pronóstico de carga de tickets
🏆 Ranking de categorías por impacto
📈 Análisis de comportamiento
🤖 Consulta al Agente IA con herramientas
━━━━━━━━━━━━━━━━━━━━━━━━━
"""

# ── Teclados interactivos ─────────────────────────────────────────────────────
def _kb(*filas):
    mk = InlineKeyboardMarkup()
    for fila in filas:
        mk.add(*[InlineKeyboardButton(txt, callback_data=dat) for txt, dat in fila])
    return mk

def _kb_lista(items, prefix, cols=2):
    mk = InlineKeyboardMarkup(row_width=cols)
    mk.add(*[InlineKeyboardButton(i, callback_data=f"{prefix}{i}") for i in items])
    return mk

def _kb_menu_usuario():
    return _kb(
        [("🆕 Reportar incidente",    "menu__reportar")],
        [("✏️ Modificar mi ticket",   "menu__modificar")],
        [("🗑 Eliminar ticket",        "menu__eliminar")],
        [("🎫 Ver mi ticket",          "menu__mi_ticket")],
        [("🤖 Consulta al Agente IA",  "menu__agente")],
    )

def _kb_menu_admin():
    return _kb(
        [("🔄 Actualizar ticket",       "menu__actualizar")],
        [("🔍 Consultar tickets",        "menu__consultar")],
        [("📊 Gráficos estadísticos",    "menu__grafico")],
        [("🔮 Pronóstico de carga",      "menu__pronostico")],
        [("🏆 Ranking de categorías",    "menu__ranking")],
        [("📈 Análisis comportamiento",  "menu__comportamiento")],
        [("🤖 Consulta al Agente IA",    "menu__agente")],
    )

# ── Utilidades de sesión ──────────────────────────────────────────────────────
def _rol(cid):       return user_roles.get(cid, "")
def _es_admin(cid):  return user_roles.get(cid) == "Administrador TI"
def _en_sesion(cid): return cid in user_sessions

def _bloquear(cid) -> bool:
    if _en_sesion(cid):
        bot.send_message(cid,
            "⚠️ Tienes una operación en curso.\n"
            "Completa los pasos o escribe /cancelar para salir.")
        return True
    return False

def _nuevo_ticket(cid) -> int:
    global next_ticket_id
    tid = next_ticket_id
    next_ticket_id += 1
    tickets[tid] = {
        "id": tid, "chat_id": cid, "estado": "Pendiente",
        "fecha": datetime.now(), "prioridad": "Normal",
        "categoria": "", "subcategoria": "", "descripcion": "",
        "ambiente": "", "aula": "", "rol_creador": _rol(cid),
        "tiempo_resolucion": None, "resumen_ia": "Pendiente análisis",
    }
    return tid

def _cerrar_reporte(cid, tid):
    # Clasificar automáticamente con Gemini
    t = tickets[tid]
    if t.get("descripcion"):
        bot.send_message(cid, "🧠 Analizando tu incidente con IA...")
        clasif = clasificar_ticket_con_ia(t["descripcion"])
        # Actualizar con sugerencias de IA si el usuario no eligió
        if not t.get("categoria"):
            tickets[tid]["categoria"]    = clasif.get("categoria", "Otros / General")
        if not t.get("subcategoria"):
            tickets[tid]["subcategoria"] = clasif.get("subcategoria", "Consulta general")
        tickets[tid]["resumen_ia"] = clasif.get("resumen", "-")

    buf = _png_ticket(tickets[tid])
    bot.send_photo(cid, buf,
        caption=(
            f"✅ Ticket #{tid} registrado correctamente!\n\n"
            f"🧠 Clasificación IA: {tickets[tid].get('resumen_ia', '-')}\n"
            f"Usa /menu para continuar o /mi_ticket para consultar."
        ))
    user_sessions.pop(cid, None)
    _mostrar_menu(cid)

def _mostrar_menu(cid):
    if _es_admin(cid):
        bot.send_message(cid, "Selecciona una acción del panel de Administrador:",
                         reply_markup=_kb_menu_admin())
    else:
        bot.send_message(cid, "Selecciona una acción del menú:",
                         reply_markup=_kb_menu_usuario())

print("✅ Módulo de teclados y sesiones cargado.")

✅ Módulo de teclados y sesiones cargado.


## Celda 12 — Comandos del Bot

In [17]:
# ── Comandos Telegram ─────────────────────────────────────────────────────────

@bot.message_handler(commands=["start"])
def cmd_start(msg):
    cid = msg.chat.id
    user_sessions.pop(cid, None)
    user_roles.pop(cid, None)
    chat_histories.pop(cid, None)
    bot.send_message(cid,
        "👋 Bienvenido al Asistente de IA del Helpdesk DTI-UDEP!\n\n"
        "Soy un agente inteligente que combina herramientas de IA para ayudarte "
        "a registrar y resolver incidentes tecnológicos.\n\n"
        "🎙️ También puedo entender mensajes de voz.\n\n"
        "Para comenzar, ¿cuál es tu rol?",
        reply_markup=_kb_lista(ROLES, "rol__", cols=1)
    )

@bot.message_handler(commands=["menu"])
def cmd_menu(msg):
    cid = msg.chat.id
    if not _rol(cid):
        bot.send_message(cid, "Primero identifícate con /start.")
        return
    _mostrar_menu(cid)

@bot.message_handler(commands=["cancelar"])
def cmd_cancelar(msg):
    cid = msg.chat.id
    user_sessions.pop(cid, None)
    bot.send_message(cid,
        "🔄 Operación cancelada.\n\nEscribe /menu para ver las opciones o /start para reiniciar.")

@bot.message_handler(commands=["ia", "agente"])
def cmd_ia(msg):
    """Consulta directa al agente IA (sin flujo de ticket)."""
    cid  = msg.chat.id
    text = msg.text.split(" ", 1)
    if len(text) < 2 or not text[1].strip():
        bot.send_message(cid,
            "🤖 Agente IA activo.\n"
            "Escribe tu consulta después del comando.\n\n"
            "Ejemplos:\n"
            "  /ia ¿Cuántos tickets urgentes hay hoy?\n"
            "  /ia ¿Cuál es el clima en Piura?\n"
            "  /ia ¿Cuánto es 500 USD en soles?")
        return
    pregunta = text[1].strip()
    bot.send_message(cid, "🔄 Consultando al agente IA...")
    respuesta = consulta_agente_openrouter(pregunta)
    bot.send_message(cid, f"🤖 {respuesta}")

@bot.message_handler(commands=["pronostico"])
def cmd_pronostico(msg):
    if not _es_admin(msg.chat.id):
        bot.send_message(msg.chat.id, "🚫 Solo para Administradores TI.")
        return
    _enviar_pronostico(msg.chat.id)

@bot.message_handler(commands=["ranking"])
def cmd_ranking(msg):
    if not _es_admin(msg.chat.id):
        bot.send_message(msg.chat.id, "🚫 Solo para Administradores TI.")
        return
    _enviar_ranking(msg.chat.id)

@bot.message_handler(regexp=r"^/")
def cmd_desconocido(msg):
    if _en_sesion(msg.chat.id): return
    bot.send_message(msg.chat.id,
        "❓ Comando no reconocido. Escribe /menu para ver las opciones.")

print("✅ Comandos del bot registrados.")

✅ Comandos del bot registrados.


## Celda 13 — Manejadores de Audio y Texto (Máquina de Estados)

In [18]:
# ── Manejador de Audio / Mensajes de Voz ─────────────────────────────────────
@bot.message_handler(content_types=["voice", "audio"])
def manejar_audio(msg):
    """
    Procesa mensajes de voz y archivos de audio:
    1. Descarga el archivo desde Telegram
    2. Transcribe con Gemini 1.5 Flash
    3. Si hay sesión activa → usa la transcripción como entrada de texto
    4. Si no hay sesión → consulta al agente IA con el texto transcrito
    """
    cid = msg.chat.id
    if not _rol(cid):
        bot.send_message(cid, "Por favor usa /start primero.")
        return

    bot.send_message(cid, "🎙️ Recibí tu audio. Transcribiendo con IA...")

    try:
        # Obtener file_id según tipo de mensaje
        if msg.content_type == "voice":
            file_info = bot.get_file(msg.voice.file_id)
        else:
            file_info = bot.get_file(msg.audio.file_id)

        # Descargar audio
        file_bytes = bot.download_file(file_info.file_path)

        # Transcribir con Gemini
        transcripcion = transcribir_audio_gemini(file_bytes, mime_type="audio/ogg")

        if "[Error" in transcripcion or "[Audio sin" in transcripcion:
            bot.send_message(cid,
                f"⚠️ No pude transcribir el audio.\n{transcripcion}\n"
                "Por favor envía un mensaje de texto.")
            return

        bot.send_message(cid, f"📝 Transcripción:\n\n_{transcripcion}_")

        ses = user_sessions.get(cid, {})
        flujo = ses.get("flujo")

        if flujo == "reportar" and ses.get("paso") == "descripcion":
            # Si está en flujo de creación de ticket, usar como descripción
            tid = ses["tid"]
            tickets[tid]["descripcion"] = transcripcion
            bot.send_message(cid,
                "✅ Descripción registrada desde tu audio.\n"
                "¿En qué edificio o ambiente ocurrió el problema?",
                reply_markup=_kb_lista(AMBIENTES, "ramb__", cols=2))
            ses["paso"] = "ambiente"
        elif flujo == "agente_ia":
            # Si está en modo agente, procesar con el LLM
            user_sessions.pop(cid, None)
            bot.send_message(cid, "🔄 Procesando tu consulta con el Agente IA...")
            respuesta = consulta_agente_openrouter(transcripcion)
            bot.send_message(cid, f"🤖 {respuesta}")
            _mostrar_menu(cid)
        else:
            # Sin sesión → respuesta conversacional directa con Gemini
            bot.send_message(cid, "🧠 Consultando al asistente IA...")
            respuesta = respuesta_gemini_conversacional(cid, transcripcion)
            bot.send_message(cid, f"🤖 {respuesta}")

    except Exception as e:
        bot.send_message(cid, f"❌ Error procesando el audio: {str(e)[:100]}")


# ── Manejador de Texto ────────────────────────────────────────────────────────
@bot.message_handler(func=lambda m: True, content_types=["text"])
def manejar_texto(msg):
    cid   = msg.chat.id
    texto = msg.text.strip()

    if cid not in user_sessions:
        if not _rol(cid):
            bot.send_message(cid, "Usa /start para comenzar.")
        else:
            # Sin sesión activa → respuesta conversacional con Gemini
            bot.send_message(cid, "🧠 Consultando al asistente...")
            respuesta = respuesta_gemini_conversacional(cid, texto)
            bot.send_message(cid, f"🤖 {respuesta}\n\n_Escribe /menu para ver las opciones._")
        return

    ses   = user_sessions[cid]
    flujo = ses.get("flujo")

    # ── Clave de administrador ────────────────────────────────────────────────
    if ses.get("esperando") == "clave_admin":
        if texto == ADMIN_CODE:
            user_roles[cid] = "Administrador TI"
            user_sessions.pop(cid)
            bot.send_message(cid, BIENVENIDA_ADMIN)
            _mostrar_menu(cid)
        else:
            bot.send_message(cid, "❌ Clave incorrecta. Intenta de nuevo o /start para cambiar de rol.")
        return

    # ── Flujo: agente IA libre ────────────────────────────────────────────────
    if flujo == "agente_ia":
        user_sessions.pop(cid, None)
        bot.send_message(cid, "🔄 Consultando al Agente IA con herramientas...")
        respuesta = consulta_agente_openrouter(texto)
        bot.send_message(cid, f"🤖 {respuesta}")
        _mostrar_menu(cid)
        return

    # ── Flujo: reportar ───────────────────────────────────────────────────────
    if flujo == "reportar":
        paso = ses.get("paso")
        tid  = ses["tid"]
        if paso == "descripcion":
            tickets[tid]["descripcion"] = texto
            bot.send_message(cid, "¿En qué edificio o ambiente ocurrió el problema?",
                             reply_markup=_kb_lista(AMBIENTES, "ramb__", cols=2))
            ses["paso"] = "ambiente"
        elif paso == "aula":
            tickets[tid]["aula"] = texto
            if _rol(cid) in ("Profesor", "Personal Administrativo"):
                bot.send_message(cid, "¿Cuál es el nivel de urgencia?",
                                 reply_markup=_kb_lista(PRIORIDADES, "rpri__", cols=2))
                ses["paso"] = "prioridad"
            else:
                _cerrar_reporte(cid, tid)
        return

    # ── Flujo: modificar ──────────────────────────────────────────────────────
    if flujo == "modificar":
        paso = ses.get("paso")
        if paso == "pedir_id":
            try: tid = int(texto)
            except ValueError:
                bot.send_message(cid, "❌ Ingresa solo el número (ej: 1005)."); return
            if tid not in tickets:
                bot.send_message(cid, f"❌ No se encontró el ticket #{tid}."); return
            if tickets[tid].get("chat_id") != cid:
                bot.send_message(cid, "🚫 Solo puedes modificar tus propios tickets."); return
            ses["tid"] = tid
            t = tickets[tid]
            campos = ["Categoria", "Subcategoria", "Descripcion", "Ambiente", "Aula"]
            if _rol(cid) in ("Profesor", "Personal Administrativo"): campos.append("Prioridad")
            bot.send_message(cid,
                f"Ticket #{tid} | Estado: {t['estado']} | Prioridad: {t['prioridad']}\n"
                f"Categoría: {t['categoria']}\n\n¿Qué campo deseas modificar?",
                reply_markup=_kb_lista(campos, "mmod__", cols=2))
            ses["paso"] = "campo"
        elif paso == "editar_valor":
            mapa = {"Categoria": "categoria", "Subcategoria": "subcategoria",
                    "Descripcion": "descripcion", "Ambiente": "ambiente",
                    "Aula": "aula", "Prioridad": "prioridad"}
            tickets[ses["tid"]][mapa[ses["campo"]]] = texto
            bot.send_message(cid, f"✅ Campo '{ses['campo']}' actualizado.")
            user_sessions.pop(cid, None)
        return

    # ── Flujo: actualizar, consultar, mi_ticket, eliminar (mismo que bot original) ─
    if flujo == "actualizar":
        try: tid = int(texto)
        except ValueError:
            bot.send_message(cid, "❌ Número inválido."); return
        if tid not in tickets:
            bot.send_message(cid, f"❌ No encontré el ticket #{tid}."); return
        ses["tid"] = tid
        t = tickets[tid]
        bot.send_message(cid,
            f"📋 Ticket #{tid}\n"
            f"Estado: {t['estado']} | Prioridad: {t['prioridad']}\n"
            f"Categoría: {t['categoria']}\n"
            f"Descripción: {t['descripcion']}\n"
            f"Ambiente: {t['ambiente']} | {t['aula']}\n\n"
            "Selecciona el nuevo estado:",
            reply_markup=_kb(
                [("⏳ Pendiente",   "aest__Pendiente")],
                [("🔧 En proceso",  "aest__En proceso")],
                [("✅ Atendido",    "aest__Atendido")],
            ))
        ses["paso"] = "seleccionar_estado"
        return

    if flujo == "consultar" and ses.get("paso") == "pedir_id":
        try: tid = int(texto)
        except ValueError:
            bot.send_message(cid, "❌ Número inválido."); return
        if tid not in tickets:
            bot.send_message(cid, f"❌ No encontré el ticket #{tid}."); return
        buf = _png_ticket(tickets[tid])
        bot.send_photo(cid, buf, caption=f"📋 Comprobante del Ticket #{tid}")
        user_sessions.pop(cid, None)
        _mostrar_menu(cid)
        return

    if flujo == "mi_ticket":
        try: tid = int(texto)
        except ValueError:
            bot.send_message(cid, "❌ Número inválido."); return
        if tid not in tickets:
            bot.send_message(cid, f"❌ No encontré el ticket #{tid}.\n(Recuerda: los IDs empiezan en 1000)"); return
        buf = _png_ticket(tickets[tid])
        bot.send_photo(cid, buf, caption=f"🎫 Tu Ticket #{tid}")
        user_sessions.pop(cid, None)
        _mostrar_menu(cid)
        return

    if flujo == "eliminar" and ses.get("paso") == "pedir_id":
        try: tid = int(texto)
        except ValueError:
            bot.send_message(cid, "❌ Número inválido."); return
        if tid not in tickets:
            bot.send_message(cid, f"❌ No encontré el ticket #{tid}."); return
        if tickets[tid].get("chat_id") != cid:
            bot.send_message(cid, "🚫 Solo puedes eliminar tus propios tickets."); return
        if tickets[tid]["estado"] == "Atendido":
            bot.send_message(cid, f"🚫 El ticket #{tid} ya fue atendido y no puede eliminarse.")
            user_sessions.pop(cid, None); return
        ses["tid"] = tid; ses["paso"] = "confirmar"
        t = tickets[tid]
        bot.send_message(cid,
            f"⚠️ ¿Eliminar el ticket #{tid}?\n"
            f"  Descripción: {t['descripcion']}\n"
            f"  Estado: {t['estado']}\n\nEsta acción no se puede deshacer.",
            reply_markup=_kb(
                [("🗑 Sí, eliminar", "del__confirmar")],
                [("↩️ Cancelar",     "del__cancelar")],
            ))
        return

print("✅ Manejadores de audio y texto registrados.")

✅ Manejadores de audio y texto registrados.


## Celda 14 — Funciones de Análisis para Admin

In [19]:
# ── Funciones de análisis enviadas por el bot ─────────────────────────────────

def _enviar_pronostico(cid):
    """Envía pronóstico de carga como texto + gráfico."""
    bot.send_message(cid, "🔮 Generando pronóstico...")

    # Texto
    texto_pronostico = forecast_tool("")
    bot.send_message(cid, texto_pronostico)

    # Tabla
    df_forecast = pronostico_tickets_prox_dias(7)
    if not df_forecast.empty:
        tabla_str = "📅 Próximos 7 días:\n"
        tabla_str += "━━━━━━━━━━━━━━━━━━━━━\n"
        for _, row in df_forecast.iterrows():
            barra = "█" * int(row["tickets_esperados"])
            tabla_str += f"  {row['fecha']}: {row['tickets_esperados']:2d} tickets {barra[:10]}\n"
        bot.send_message(cid, tabla_str)

    # Gráfico
    buf = _grafico_pronostico()
    if buf:
        bot.send_photo(cid, buf, caption="🔮 Pronóstico — Próximos 7 días")
    _mostrar_menu(cid)


def _enviar_ranking(cid):
    """Envía ranking de categorías + gráfico."""
    bot.send_message(cid, "🏆 Generando ranking...")

    ranking = ranking_categorias()
    msg_str = "🏆 RANKING DE CATEGORÍAS POR IMPACTO:\n━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    for i, (cat, datos) in enumerate(ranking[:8], 1):
        emoji = ["🥇","🥈","🥉","4️⃣","5️⃣","6️⃣","7️⃣","8️⃣"][i-1]
        msg_str += (
            f"{emoji} {cat}\n"
            f"   Total: {datos['total']} | Urgentes: {datos['urgentes']} | "
            f"Resueltos: {datos['pct_resolucion']}%\n"
        )

    bot.send_message(cid, msg_str)

    buf = _grafico_ranking()
    bot.send_photo(cid, buf, caption="🏆 Ranking de Categorías por Impacto")
    _mostrar_menu(cid)


def _enviar_comportamiento(cid):
    """Envía análisis de comportamiento + gráfico de horas pico."""
    bot.send_message(cid, "📈 Analizando comportamiento...")

    analisis = analisis_comportamiento()
    if not analisis:
        bot.send_message(cid, "Insuficientes datos para análisis.")
        return

    por_rol_str = " | ".join([f"{k}: {v}" for k, v in analisis['por_rol'].items()])
    top_cats_str = "\n".join([f"  • {k}: {v} tickets" for k, v in list(analisis['top_categorias'].items())[:5]])

    msg_str = (
        f"📈 ANÁLISIS DE COMPORTAMIENTO\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"📌 Total tickets: {analisis['total']}\n"
        f"⏰ Hora pico: {analisis['hora_pico']:02d}:00h\n"
        f"📅 Día más activo: {analisis['dia_pico']}\n"
        f"✅ Tasa de resolución: {analisis['tasa_resolucion']}%\n"
        f"⏱️ Tiempo prom. resolución: {analisis['tiempo_prom_resolucion_min']:.0f} min\n"
        f"🚨 Críticos pendientes: {analisis['criticos_pendientes']}\n"
        f"\n👥 Por rol:\n  {por_rol_str}\n"
        f"\n🏷️ Top categorías:\n{top_cats_str}\n"
    )

    bot.send_message(cid, msg_str)
    buf = _grafico_horas_pico()
    bot.send_photo(cid, buf, caption="⏰ Distribución de Tickets por Hora del Día")
    _mostrar_menu(cid)

print("✅ Funciones de análisis para administrador listas.")

✅ Funciones de análisis para administrador listas.


## Celda 15 — Manejador de Callbacks

In [20]:
# ── Manejador central de callbacks ────────────────────────────────────────────
@bot.callback_query_handler(func=lambda c: True)
def cb_handler(call):
    cid  = call.message.chat.id
    data = call.data
    bot.answer_callback_query(call.id)
    ses = user_sessions.get(cid, {})

    # ── Selección de rol ──────────────────────────────────────────────────────
    if data.startswith("rol__"):
        rol = data[5:]
        if rol == "Administrador TI":
            user_sessions[cid] = {"esperando": "clave_admin"}
            bot.send_message(cid,
                "🔐 Acceso restringido.\nIngresa tu clave de 6 dígitos:")
        else:
            user_roles[cid] = rol
            bot.send_message(cid, BIENVENIDA_USUARIO.format(rol=rol))
            _mostrar_menu(cid)
        return

    # ── Menú principal ────────────────────────────────────────────────────────
    if data == "menu__reportar":
        if _es_admin(cid):
            bot.send_message(cid, "Los administradores no reportan incidentes."); return
        if _bloquear(cid): return
        _iniciar_reporte(cid)
        return
    if data == "menu__modificar":
        if _bloquear(cid): return
        user_sessions[cid] = {"flujo": "modificar", "paso": "pedir_id"}
        bot.send_message(cid, "✏️ Ingresa el número de ticket a modificar (ej: 1005):")
        return
    if data == "menu__eliminar":
        if _bloquear(cid): return
        user_sessions[cid] = {"flujo": "eliminar", "paso": "pedir_id"}
        bot.send_message(cid, "🗑 Ingresa el número de ticket a eliminar (ej: 1005):")
        return
    if data == "menu__mi_ticket":
        if _bloquear(cid): return
        user_sessions[cid] = {"flujo": "mi_ticket"}
        bot.send_message(cid, "🎫 Ingresa el número de tu ticket (ej: 1002):")
        return
    if data == "menu__actualizar":
        if _bloquear(cid): return
        user_sessions[cid] = {"flujo": "actualizar", "paso": "pedir_id"}
        bot.send_message(cid, "🔄 Ingresa el número de ticket a actualizar:")
        return
    if data == "menu__consultar":
        if _bloquear(cid): return
        user_sessions[cid] = {"flujo": "consultar"}
        bot.send_message(cid, "🔍 ¿Cómo deseas consultar?",
            reply_markup=_kb(
                [("🎫 Ticket individual",   "cons__individual")],
                [("📄 Reporte general PDF", "cons__reporte")],
            ))
        return
    if data == "menu__grafico":
        bot.send_message(cid, "📊 Tipo de visualización:",
            reply_markup=_kb(
                [("🥧 Gráfico circular",   "graf__pie")],
                [("📈 Línea temporal",     "graf__time")],
                [("⏰ Horas pico",         "graf__horas")],
            ))
        return
    if data == "menu__pronostico":
        if not _es_admin(cid):
            bot.send_message(cid, "🚫 Solo para Administradores TI."); return
        _enviar_pronostico(cid); return
    if data == "menu__ranking":
        if not _es_admin(cid):
            bot.send_message(cid, "🚫 Solo para Administradores TI."); return
        _enviar_ranking(cid); return
    if data == "menu__comportamiento":
        if not _es_admin(cid):
            bot.send_message(cid, "🚫 Solo para Administradores TI."); return
        _enviar_comportamiento(cid); return
    if data == "menu__agente":
        if _bloquear(cid): return
        user_sessions[cid] = {"flujo": "agente_ia"}
        bot.send_message(cid,
            "🤖 Agente IA activado.\n\n"
            "Puedes preguntarme sobre:\n"
            "  • Estadísticas de tickets\n"
            "  • Pronóstico de carga\n"
            "  • Clima, tipo de cambio, noticias IT\n"
            "  • Cualquier cálculo o consulta\n\n"
            "Escribe tu pregunta o envía un audio 🎙️")
        return

    # ── Flujo reportar ────────────────────────────────────────────────────────
    if data.startswith("rcat__"):
        cat = data[6:]; tid = ses.get("tid")
        if not tid: return
        tickets[tid]["categoria"] = cat
        bot.send_message(cid,
            f"Categoría: {cat}\nSelecciona el tipo específico:",
            reply_markup=_kb_lista(CATEGORIAS[cat], "rsub__", cols=1))
        ses["paso"] = "subcategoria"
        return
    if data.startswith("rsub__"):
        sub = data[6:]; tid = ses.get("tid")
        if not tid: return
        tickets[tid]["subcategoria"] = sub
        bot.send_message(cid,
            f"Subcategoría: {sub}\n\n"
            "Describe brevemente el problema\n"
            "(también puedes enviar un mensaje de voz 🎙️):")
        ses["paso"] = "descripcion"
        return
    if data.startswith("ramb__"):
        amb = data[6:]; tid = ses.get("tid")
        if not tid: return
        tickets[tid]["ambiente"] = amb
        bot.send_message(cid,
            f"Ambiente: {amb}\n\nEscribe el número o nombre del aula específica:")
        ses["paso"] = "aula"
        return
    if data.startswith("rpri__"):
        tid = ses.get("tid")
        if not tid: return
        tickets[tid]["prioridad"] = data[6:]
        _cerrar_reporte(cid, tid)
        return

    # ── Flujo modificar ───────────────────────────────────────────────────────
    if data.startswith("mmod__"):
        campo = data[6:]; ses["campo"] = campo
        if campo == "Prioridad":
            bot.send_message(cid, "Nueva prioridad:", reply_markup=_kb_lista(PRIORIDADES, "mprv__", cols=2))
        elif campo == "Ambiente":
            bot.send_message(cid, "Nuevo ambiente:",  reply_markup=_kb_lista(AMBIENTES,   "mamb__", cols=2))
        elif campo == "Categoria":
            bot.send_message(cid, "Nueva categoría:", reply_markup=_kb_lista(list(CATEGORIAS.keys()), "mcat__", cols=1))
        else:
            ses["paso"] = "editar_valor"
            bot.send_message(cid, f"Escribe el nuevo valor para '{campo}':")
        return
    if data.startswith("mprv__"):
        tickets[ses["tid"]]["prioridad"] = data[6:]
        bot.send_message(cid, "✅ Prioridad actualizada.")
        user_sessions.pop(cid, None); _mostrar_menu(cid); return
    if data.startswith("mamb__"):
        tickets[ses["tid"]]["ambiente"] = data[6:]
        bot.send_message(cid, "✅ Ambiente actualizado.")
        user_sessions.pop(cid, None); _mostrar_menu(cid); return
    if data.startswith("mcat__"):
        cat = data[6:]; tickets[ses["tid"]]["categoria"] = cat
        bot.send_message(cid, f"Categoría: {cat}. Subcategoría:",
            reply_markup=_kb_lista(CATEGORIAS[cat], "msub__", cols=1))
        return
    if data.startswith("msub__"):
        tickets[ses["tid"]]["subcategoria"] = data[6:]
        bot.send_message(cid, "✅ Subcategoría actualizada.")
        user_sessions.pop(cid, None); _mostrar_menu(cid); return

    # ── Actualizar estado ─────────────────────────────────────────────────────
    if data.startswith("aest__"):
        nuevo = data[6:]; tid = ses.get("tid")
        if not tid: return
        tickets[tid]["estado"] = nuevo
        if nuevo == "Atendido":
            tickets[tid]["tiempo_resolucion"] = int(
                (datetime.now() - tickets[tid]["fecha"]).total_seconds() / 60)
        user_sessions.pop(cid, None)
        buf = _png_ticket(tickets[tid])
        bot.send_photo(cid, buf, caption=f"✅ Ticket #{tid} → {nuevo}")
        # Notificar al creador
        creador = tickets[tid].get("chat_id")
        if creador and creador != cid and nuevo == "Atendido":
            try:
                bot.send_message(creador,
                    f"🎉 Tu ticket #{tid} ha sido atendido y resuelto.")
                bot.send_photo(creador, _png_ticket(tickets[tid]))
            except: pass
        _mostrar_menu(cid); return

    # ── Consultar ─────────────────────────────────────────────────────────────
    if data == "cons__individual":
        ses["paso"] = "pedir_id"
        bot.send_message(cid, "Ingresa el número del ticket:"); return
    if data == "cons__reporte":
        bot.send_message(cid, "Organizar reporte por:",
            reply_markup=_kb(
                [("📊 Estado",    "crep__Estado")],
                [("🗂 Tipo",      "crep__Categoria")],
                [("🏢 Ambiente",  "crep__Ambiente")],
            ))
        return
    if data.startswith("crep__"):
        criterio = data[6:]
        bot.send_message(cid, f"⏳ Generando reporte PDF por {criterio}...")
        try:
            buf = _pdf_reporte(criterio)
            bot.send_document(cid, buf,
                visible_file_name=f"reporte_DTI_{criterio.lower()}.pdf",
                caption=f"📄 Reporte por {criterio} | {len(tickets)} tickets | {datetime.now().strftime('%d/%m/%Y %H:%M')}")
        except Exception as e:
            bot.send_message(cid, f"❌ Error al generar PDF: {e}")
        user_sessions.pop(cid, None)
        _mostrar_menu(cid); return

    # ── Gráficos ──────────────────────────────────────────────────────────────
    if data == "graf__time":
        buf = _grafico_temporal()
        bot.send_photo(cid, buf, caption="📈 Evolución de tickets")
        _mostrar_menu(cid); return
    if data == "graf__horas":
        buf = _grafico_horas_pico()
        bot.send_photo(cid, buf, caption="⏰ Horas pico")
        _mostrar_menu(cid); return
    if data == "graf__pie":
        bot.send_message(cid, "Agrupar gráfico circular por:",
            reply_markup=_kb(
                [("📊 Estado",   "gpie__Estado")],
                [("🗂 Tipo",     "gpie__Categoria")],
                [("🏢 Ambiente", "gpie__Ambiente")],
            ))
        return
    if data.startswith("gpie__"):
        criterio = data[6:]
        buf = _grafico_pie(criterio)
        bot.send_photo(cid, buf, caption=f"🥧 Distribución por {criterio}")
        _mostrar_menu(cid); return

    # ── Eliminar ──────────────────────────────────────────────────────────────
    if data == "del__confirmar":
        tid = ses.get("tid")
        tickets.pop(tid, None)
        user_sessions.pop(cid, None)
        bot.send_message(cid, f"✅ Ticket #{tid} eliminado.")
        _mostrar_menu(cid); return
    if data == "del__cancelar":
        user_sessions.pop(cid, None)
        bot.send_message(cid, "↩️ Eliminación cancelada.")
        _mostrar_menu(cid); return

print("✅ Manejador de callbacks registrado.")

✅ Manejador de callbacks registrado.


## Celda 16 — Inicialización de Flujos de Reportes

In [21]:
# ── Inicialización del flujo de reporte ───────────────────────────────────────
def _iniciar_reporte(cid):
    tid = _nuevo_ticket(cid)
    user_sessions[cid] = {"flujo": "reportar", "tid": tid}
    bot.send_message(cid,
        f"📝 Creando ticket #{tid}...\n\n"
        "Selecciona el área del problema:\n"
        "_(También puedes describir el problema con un audio 🎙️ en el próximo paso)_",
        reply_markup=_kb_lista(list(CATEGORIAS.keys()), "rcat__", cols=1)
    )

print("✅ Función _iniciar_reporte registrada.")

✅ Función _iniciar_reporte registrada.


## Celda 17 — 🚀 Ejecución del Bot

In [ ]:
# ── Verificación de alertas activas al iniciar ────────────────────────────────
tickets_criticos = [
    t for t in tickets.values()
    if t["prioridad"] in ["Urgente", "Inmediata"] and t["estado"] != "Atendido"
]
if tickets_criticos:
    print(f"⚠️  ATENCIÓN: {len(tickets_criticos)} ticket(s) crítico(s) sin atender.")
    for t in tickets_criticos[:3]:
        print(f"   #{t['id']} [{t['prioridad']}] — {t['categoria']}: {t['descripcion'][:50]}")

# ── Resumen del sistema ───────────────────────────────────────────────────────
analisis_ini = analisis_comportamiento()
print("\n📊 Estado del sistema al iniciar:")
print(f"   Total tickets: {analisis_ini['total']}")
print(f"   Tasa resolución: {analisis_ini['tasa_resolucion']}%")
print(f"   Hora pico histórica: {analisis_ini['hora_pico']:02d}:00h")
print(f"   Críticos pendientes: {analisis_ini['criticos_pendientes']}")

print("\n🤖 AgenteIA DTI-UDEP iniciado.")
print("   Escucha activa en Telegram...")
print("   Funcionalidades: Audio IA | Clasificación IA | Pronóstico | Ranking | Análisis")
print("   Usa Ctrl+C para detener.")

# ── Iniciar polling ───────────────────────────────────────────────────────────
bot.infinity_polling()

⚠️  ATENCIÓN: 5 ticket(s) crítico(s) sin atender.
   #1004 [Urgente] — Plataformas Academicas: La computadora hace un ruido extrano al encenderse
   #1009 [Urgente] — Software: No puedo ingresar al sistema desde esta manana.
   #1013 [Urgente] — Otros / General: La pantalla de mi laptop tiene rayas horizontales.

📊 Estado del sistema al iniciar:
   Total tickets: 20
   Tasa resolución: 25.0%
   Hora pico histórica: 07:00h
   Críticos pendientes: 5

🤖 AgenteIA DTI-UDEP iniciado.
   Escucha activa en Telegram...
   Funcionalidades: Audio IA | Clasificación IA | Pronóstico | Ranking | Análisis
   Usa Ctrl+C para detener.
